# Discrete-Time Hazard Forecasting and Financial Risk

This notebook covers the bridge between survival analysis and time-series forecasting: expanding right-censored records into longitudinal panels, fitting discrete hazard models with LightGBM, and applying Cox PH to credit risk.

## Simulating a Loan Portfolio

We generate right-censored loan data where credit scores influence time-to-default.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(0)
n_samples = 1500

credit_score = np.random.normal(650, 50, n_samples)
baseline_hazard = 0.02
risk_multiplier = np.exp(-(credit_score - 650) / 100)
hazard = baseline_hazard * risk_multiplier
time_to_default = np.random.exponential(1 / hazard)

observation_window = 60
observed = time_to_default <= observation_window
duration = np.minimum(time_to_default, observation_window)

loans = pd.DataFrame({
    "id": range(n_samples),
    "duration": duration,
    "event": observed.astype(int),
    "credit_score": credit_score,
})

print(f"Portfolio: {n_samples} loans")
print(f"Defaulted: {loans['event'].sum()} ({loans['event'].mean():.1%})")
print(f"Censored: {(1 - loans['event']).sum()} ({1 - loans['event'].mean():.1%})")

## Cox Proportional Hazards for Credit Risk

We fit a Cox model to estimate how credit scores affect the hazard of default. A hazard ratio < 1 means the feature reduces instantaneous default risk.

In [ ]:
from lifelines import CoxPHFitter
import matplotlib.pyplot as plt

cph = CoxPHFitter(penalizer=0.1)
df_fit = loans.drop(columns=["id"])
cph.fit(df_fit, duration_col="duration", event_col="event")

cph.print_summary(decimals=4)

In [ ]:
# Extract hazard ratios
hr = cph.summary[["exp(coef)", "p"]]
print("Hazard Ratios:")
print(hr)
print("\nA hazard ratio < 1 for credit_score means higher scores reduce default risk.")

In [ ]:
# Plot survival curves for different credit score profiles
fig, ax = plt.subplots(figsize=(10, 6))

for score in [550, 650, 750]:
    profile = pd.DataFrame({"credit_score": [score]})
    cph.predict_survival_function(profile).plot(ax=ax, label=f"Score={score}")

ax.set_title("Predicted Survival by Credit Score")
ax.set_xlabel("Months")
ax.set_ylabel("Probability of Not Defaulting")
ax.legend()
plt.tight_layout()
plt.show()

## Expanding to Longitudinal Person-Period Panels

To integrate survival into standard forecasting frameworks, we expand each cross-sectional record into one row per time period at risk. The target becomes a binary indicator of whether the event occurred at that specific interval.

In [ ]:
def expand_to_longitudinal_panel(df, max_horizon=60):
    """Convert right-censored records into person-period panels."""
    panel_rows = []
    for _, row in df.iterrows():
        t_max = min(int(np.ceil(row["duration"])), max_horizon)
        for t in range(1, t_max + 1):
            is_event = 1 if (t == t_max and row["event"] == 1) else 0
            panel_rows.append({
                "unique_id": row["id"],
                "ds": t,
                "event": is_event,
                "credit_score": row["credit_score"],
            })
    return pd.DataFrame(panel_rows)


panel = expand_to_longitudinal_panel(loans)
print(f"Panel shape: {panel.shape}")
print(f"Event rate per period: {panel['event'].mean():.4f}")
panel.head(10)

## Simulating Marketing Conversion Data for Discrete Hazard

In [ ]:
# Richer marketing data for the LightGBM discrete hazard model
np.random.seed(42)
n = 3000

marketing = pd.DataFrame({
    "ad_spend": np.random.gamma(2, 40, n),
    "page_views": np.random.poisson(4, n),
    "channel": np.random.choice([0, 1], n),
})

risk = (
    0.015 * marketing["ad_spend"]
    + 0.3 * marketing["channel"]
    + 0.05 * marketing["page_views"]
    + 0.0005 * marketing["ad_spend"] * marketing["page_views"]
)

time_to_convert = np.random.exponential(1 / np.exp(risk))
obs_window = 30

marketing["duration"] = np.minimum(time_to_convert, obs_window)
marketing["event"] = (time_to_convert <= obs_window).astype(int)

print(f"Marketing dataset: {len(marketing)} users")
print(f"Converted: {marketing['event'].mean():.1%}")

## Discrete Hazard Model with LightGBM

We expand the marketing data into a panel and train a LightGBM classifier to predict per-period hazard. The cumulative product of (1 - hazard) recovers the survival curve.

In [ ]:
def expand_marketing_to_panel(df, max_horizon=30):
    """Transform right-censored records into discrete-time panels."""
    rows = []
    for idx, row in df.iterrows():
        duration = int(row["duration"])
        event = row["event"]
        limit = min(duration, max_horizon)
        for t in range(1, limit + 1):
            is_event = 1 if (event == 1 and t == duration) else 0
            rows.append({
                "unique_id": idx,
                "ds": t,
                "event": is_event,
                "ad_spend": row["ad_spend"],
                "page_views": row["page_views"],
                "channel": row["channel"],
            })
    return pd.DataFrame(rows)


marketing_panel = expand_marketing_to_panel(marketing)
print(f"Marketing panel: {marketing_panel.shape}")
marketing_panel.head(10)

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split

feature_cols = ["ds", "ad_spend", "page_views", "channel"]

# Split by unique_id to avoid leakage
unique_ids = marketing_panel["unique_id"].unique()
train_ids, test_ids = train_test_split(unique_ids, test_size=0.3, random_state=42)

train_panel = marketing_panel[marketing_panel["unique_id"].isin(train_ids)]
test_panel = marketing_panel[marketing_panel["unique_id"].isin(test_ids)]

clf = LGBMClassifier(random_state=42, n_estimators=100, verbose=-1)
clf.fit(train_panel[feature_cols], train_panel["event"])

print("LightGBM discrete hazard model trained.")

In [ ]:
# Predict hazard probabilities and recover survival curves
test_panel = test_panel.copy()
test_panel["hazard_prob"] = clf.predict_proba(test_panel[feature_cols])[:, 1]
test_panel["survival"] = (
    (1 - test_panel["hazard_prob"])
    .groupby(test_panel["unique_id"])
    .cumprod()
)

# Show survival curves for a few test users
fig, ax = plt.subplots(figsize=(10, 6))
for uid in test_ids[:5]:
    user_data = test_panel[test_panel["unique_id"] == uid]
    ax.step(user_data["ds"], user_data["survival"], where="post", label=f"User {uid}")

ax.set_title("Discrete-Time Survival Curves (LightGBM)")
ax.set_xlabel("Time Period")
ax.set_ylabel("Survival Probability")
ax.legend()
plt.tight_layout()
plt.show()

## Evaluating the Discrete Hazard Model

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss

# Per-period evaluation
auc = roc_auc_score(test_panel["event"], test_panel["hazard_prob"])
brier = brier_score_loss(test_panel["event"], test_panel["hazard_prob"])

print(f"Per-period AUC: {auc:.4f}")
print(f"Per-period Brier Score: {brier:.4f}")